# Système de Recommandation de Films
### MovieLens 100K — User-Based · Item-Based · Content-Based

## Etape 1 — Import du dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

plt.rcParams.update({
    'figure.facecolor':'#0f0f1a','axes.facecolor':'#161625',
    'text.color':'#e0e0f0','axes.labelcolor':'#e0e0f0',
    'xtick.color':'#9090b0','ytick.color':'#9090b0',
    'axes.edgecolor':'#2a2a45','grid.color':'#1e1e38'
})
ACCENT='#e50914'; BLUE='#4466ff'; GREEN='#22cc88'

ratings = pd.read_csv('data/u.data', sep='\t',
    names=['user_id','item_id','rating','timestamp'])

genre_cols = ['Action','Adventure','Animation',"Children's",'Comedy','Crime',
              'Documentary','Drama','Fantasy','Film-Noir','Horror','Musical',
              'Mystery','Romance','Sci-Fi','Thriller','War','Western']

movies = pd.read_csv('data/u.item', sep='|', encoding='latin-1',
    names=['item_id','title','release_date','video_date','imdb_url',
           'unknown']+genre_cols, usecols=range(24))

movies['year'] = pd.to_datetime(movies['release_date'], errors='coerce').dt.year
movies['genres'] = movies[genre_cols].apply(
    lambda r: [g for g in genre_cols if r[g]==1], axis=1)
movies['genres_str'] = movies['genres'].apply(lambda g: ', '.join(g) if g else 'Unknown')

print(f'{len(ratings):,} evaluations | {ratings.user_id.nunique()} utilisateurs | {ratings.item_id.nunique()} films')
movies[['item_id','title','year','genres_str']].head(5)

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(15,4))
fig.suptitle("Vue d'ensemble MovieLens 100K", color='#e0e0f0')

nc = ratings['rating'].value_counts().sort_index()
axes[0].bar(nc.index, nc.values, color=ACCENT, alpha=0.85)
axes[0].set_title('Distribution des notes', color='#e0e0f0')

gc = pd.Series({g: movies[g].sum() for g in genre_cols}).sort_values().tail(10)
axes[1].barh(gc.index, gc.values, color=BLUE, alpha=0.8)
axes[1].set_title('Top 10 Genres', color='#e0e0f0')

axes[2].hist(ratings.groupby('user_id').size(), bins=30, color=GREEN, alpha=0.8)
axes[2].set_title('Evaluations / utilisateur', color='#e0e0f0')

plt.tight_layout(); plt.show()

## Etape 2 — Matrice User-Movie

In [ ]:
user_movie_matrix = ratings.pivot_table(index='user_id', columns='item_id', values='rating')
matrix_filled = user_movie_matrix.fillna(0)
sparsity = 1 - (len(ratings) / (user_movie_matrix.shape[0] * user_movie_matrix.shape[1]))
print(f'Matrice : {user_movie_matrix.shape} | Sparsite : {sparsity:.1%}')
user_movie_matrix.iloc[:5,:8]

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
sns.heatmap(user_movie_matrix.iloc[:20,:20].fillna(0), cmap='Reds',
            linewidths=0.3, linecolor='#0a0a15', ax=ax, vmin=0, vmax=5)
ax.set_title('Matrice User-Movie (20x20)', color='#e0e0f0')
plt.tight_layout(); plt.show()

## Etape 3a — User-Based Filtering (Cosine Similarity)

In [ ]:
user_sim_df = pd.DataFrame(
    cosine_similarity(matrix_filled),
    index=user_movie_matrix.index, columns=user_movie_matrix.index)

def user_based_recommend(user_id, n=5, n_sim=20):
    seen = user_movie_matrix.loc[user_id].dropna().index.tolist()
    neighbors = user_sim_df[user_id].drop(user_id).sort_values(ascending=False).head(n_sim)
    scores = {}
    for item in [c for c in user_movie_matrix.columns if c not in seen]:
        col = user_movie_matrix[item]
        v = neighbors[neighbors.index.isin(col.dropna().index)]
        if v.empty or v.sum()==0: continue
        scores[item] = np.dot(v.values, col[v.index].values) / v.sum()
    top = pd.Series(scores).sort_values(ascending=False).head(n)
    res = movies[movies.item_id.isin(top.index)][['item_id','title','genres_str']].copy()
    res['note_predite'] = res.item_id.map(top)
    return res.sort_values('note_predite', ascending=False).reset_index(drop=True)

TARGET_USER = 1
reco_ub = user_based_recommend(TARGET_USER)
print(f'Top-5 User-Based pour User {TARGET_USER}')
display(reco_ub[['title','genres_str','note_predite']])

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,5))
sns.heatmap(user_sim_df.iloc[:15,:15], cmap='Blues', ax=axes[0],
            linewidths=0.2, linecolor='#0a0a15', vmin=0, vmax=1)
axes[0].set_title('Similarite cosinus — Utilisateurs', color='#e0e0f0')

bars = axes[1].barh(reco_ub['title'].str[:35], reco_ub['note_predite'], color=ACCENT, alpha=0.85)
axes[1].set_xlim(0,5.5)
axes[1].set_title(f'Top-5 User-Based (User {TARGET_USER})', color='#e0e0f0')
for b in bars:
    axes[1].text(b.get_width()+0.05, b.get_y()+b.get_height()/2,
                 f'{b.get_width():.2f}', va='center', fontsize=9)
plt.tight_layout(); plt.show()

## Etape 3b — Item-Based Filtering (Cosine Similarity)

In [ ]:
item_sim_df = pd.DataFrame(
    cosine_similarity(matrix_filled.T),
    index=user_movie_matrix.columns, columns=user_movie_matrix.columns)

def item_based_recommend(user_id, n=5, n_sim=10):
    user_ratings = user_movie_matrix.loc[user_id].dropna()
    seen = user_ratings.index.tolist()
    scores = {}
    for item in [c for c in user_movie_matrix.columns if c not in seen]:
        if item not in item_sim_df.index: continue
        sims = item_sim_df[item][seen].sort_values(ascending=False).head(n_sim)
        if sims.sum()==0: continue
        scores[item] = np.dot(sims.values, user_ratings[sims.index].values) / sims.sum()
    top = pd.Series(scores).sort_values(ascending=False).head(n)
    res = movies[movies.item_id.isin(top.index)][['item_id','title','genres_str']].copy()
    res['note_predite'] = res.item_id.map(top)
    return res.sort_values('note_predite', ascending=False).reset_index(drop=True)

reco_ib = item_based_recommend(TARGET_USER)
print(f'Top-5 Item-Based pour User {TARGET_USER}')
display(reco_ib[['title','genres_str','note_predite']])

ref_id = 1
titre_ref = movies[movies.item_id==ref_id]['title'].values[0]
sims_ref = item_sim_df[ref_id].drop(ref_id).sort_values(ascending=False).head(5)
sim_films = movies[movies.item_id.isin(sims_ref.index)][['title','genres_str']].copy()
sim_films.index = range(len(sim_films))
sim_films['similarite'] = sims_ref.values[:len(sim_films)]
print(f'\nFilms similaires a "{titre_ref}" :')
display(sim_films)

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,5))
sns.heatmap(item_sim_df.iloc[:15,:15], cmap='Purples', ax=axes[0],
            linewidths=0.2, linecolor='#0a0a15', vmin=0, vmax=1)
axes[0].set_title('Similarite cosinus — Films', color='#e0e0f0')

bars = axes[1].barh(reco_ib['title'].str[:35], reco_ib['note_predite'], color=BLUE, alpha=0.85)
axes[1].set_xlim(0,5.5)
axes[1].set_title(f'Top-5 Item-Based (User {TARGET_USER})', color='#e0e0f0')
for b in bars:
    axes[1].text(b.get_width()+0.05, b.get_y()+b.get_height()/2,
                 f'{b.get_width():.2f}', va='center', fontsize=9)
plt.tight_layout(); plt.show()

## Etape 4 — Content-Based (Genres)

In [ ]:
genre_matrix = movies.set_index('item_id')[genre_cols].fillna(0)

def content_based_recommend(user_id, n=5):
    user_ratings = user_movie_matrix.loc[user_id].dropna()
    liked = user_ratings[user_ratings>=4].index.tolist()
    if not liked: liked = user_ratings.index.tolist()
    liked_in = [i for i in liked if i in genre_matrix.index]
    if not liked_in: return pd.DataFrame()
    profile = genre_matrix.loc[liked_in].mean().values.reshape(1,-1)
    sims = pd.Series(cosine_similarity(profile, genre_matrix)[0], index=genre_matrix.index)
    sims = sims.drop([i for i in user_ratings.index if i in sims.index])
    top = sims.sort_values(ascending=False).head(n)
    res = movies[movies.item_id.isin(top.index)][['item_id','title','genres_str']].copy()
    res['score'] = res.item_id.map(top)
    return res.sort_values('score', ascending=False).reset_index(drop=True)

reco_cb = content_based_recommend(TARGET_USER)
print(f'Top-5 Content-Based pour User {TARGET_USER}')
display(reco_cb[['title','genres_str','score']])

liked_idx = user_movie_matrix.loc[TARGET_USER].dropna()
liked_idx = liked_idx[liked_idx>=4].index
profile_user = genre_matrix[genre_matrix.index.isin(liked_idx)].mean()
print(f'\nGenres preferes de l utilisateur {TARGET_USER} :')
print(profile_user[profile_user>0].sort_values(ascending=False))

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,5))
p = profile_user[profile_user>0].sort_values()
axes[0].barh(p.index, p.values, color=GREEN, alpha=0.8)
axes[0].set_title(f'Profil genres — User {TARGET_USER}', color='#e0e0f0')
axes[0].set_xlim(0,1.1)

bars = axes[1].barh(reco_cb['title'].str[:35], reco_cb['score'], color=GREEN, alpha=0.8)
axes[1].set_xlim(0,1.3)
axes[1].set_title(f'Top-5 Content-Based (User {TARGET_USER})', color='#e0e0f0')
for b in bars:
    axes[1].text(b.get_width()+0.01, b.get_y()+b.get_height()/2,
                 f'{b.get_width():.3f}', va='center', fontsize=9)
plt.tight_layout(); plt.show()

## Etape 5 — Top-5 pour un utilisateur donne

In [ ]:
USER_ID = 1  # changer ici (1-943)

hist = ratings[ratings.user_id==USER_ID].merge(
    movies[['item_id','title','genres_str']], on='item_id'
).sort_values('rating', ascending=False)

print(f'Utilisateur {USER_ID} — {len(hist)} films | Moyenne : {hist.rating.mean():.2f}/5')
print('\nFilms deja vus (top 5) :')
display(hist[['title','rating','genres_str']].head(5))

r_ub = user_based_recommend(USER_ID)
r_ib = item_based_recommend(USER_ID)
r_cb = content_based_recommend(USER_ID)

print('\n--- User-Based ---')
display(r_ub[['title','genres_str','note_predite']])
print('\n--- Item-Based ---')
display(r_ib[['title','genres_str','note_predite']])
print('\n--- Content-Based ---')
display(r_cb[['title','genres_str','score']])

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(18,5))
fig.suptitle(f'Top-5 Recommandations — Utilisateur {USER_ID}', color='#e0e0f0', fontsize=13)
for ax,(df,col,label,color) in zip(axes,[
    (r_ub,'note_predite','User-Based',ACCENT),
    (r_ib,'note_predite','Item-Based',BLUE),
    (r_cb,'score','Content-Based',GREEN)]):
    if not df.empty:
        bars = ax.barh(df['title'].str[:30], df[col], color=color, alpha=0.85)
        for b in bars:
            ax.text(b.get_width()+0.02, b.get_y()+b.get_height()/2,
                    f'{b.get_width():.2f}', va='center', fontsize=8)
    ax.set_title(label, color='#e0e0f0')
    ax.invert_yaxis()
plt.tight_layout(); plt.show()

## Etape 6 — Evaluation RMSE + Precision + Comparaison

In [ ]:
train_data, test_data = train_test_split(ratings, test_size=0.2, random_state=42)
train_mat = train_data.pivot_table(index='user_id', columns='item_id', values='rating').fillna(0)
usr_sim_tr = pd.DataFrame(cosine_similarity(train_mat), index=train_mat.index, columns=train_mat.index)
itm_sim_tr = pd.DataFrame(cosine_similarity(train_mat.T), index=train_mat.columns, columns=train_mat.columns)
print(f'Train : {len(train_data):,} | Test : {len(test_data):,}')

In [ ]:
def pred_ub(uid, iid, n=20):
    if uid not in usr_sim_tr.index or iid not in train_mat.columns: return None
    nb = usr_sim_tr[uid].drop(uid).sort_values(ascending=False).head(n)
    col = train_mat[iid]
    v = nb[nb.index.isin(col[col>0].index)]
    if v.empty or v.sum()==0: return None
    return np.dot(v.values, col[v.index].values) / v.sum()

def pred_ib(uid, iid, n=10):
    if uid not in train_mat.index or iid not in itm_sim_tr.index: return None
    rated = train_mat.loc[uid]; rated = rated[rated>0]
    sims = itm_sim_tr[iid][rated.index].sort_values(ascending=False).head(n)
    if sims.sum()==0: return None
    return np.dot(sims.values, rated[sims.index].values) / sims.sum()

def pred_cb(uid, iid):
    if uid not in train_mat.index or iid not in genre_matrix.index: return None
    rated = train_mat.loc[uid]
    liked = rated[rated>=4].index
    if len(liked)==0: liked = rated[rated>0].index
    liked_in = [i for i in liked if i in genre_matrix.index]
    if not liked_in: return None
    profile = genre_matrix.loc[liked_in].mean().values.reshape(1,-1)
    return 1 + cosine_similarity(profile, genre_matrix.loc[[iid]])[0][0] * 4

print('Calcul en cours (2000 paires)...')
sample = test_data.sample(n=2000, random_state=42)
ub_p, ib_p, cb_p, act = [], [], [], []
for _, row in sample.iterrows():
    uid, iid, r = int(row.user_id), int(row.item_id), row.rating
    p1,p2,p3 = pred_ub(uid,iid), pred_ib(uid,iid), pred_cb(uid,iid)
    if p1 and p2 and p3:
        ub_p.append(p1); ib_p.append(p2); cb_p.append(p3); act.append(r)

act = np.array(act)
rmse = lambda p: np.sqrt(mean_squared_error(act, p))
prec = lambda p,t=0.5: sum(abs(a-x)<=t for a,x in zip(act,p))/len(act)

r_bl=rmse([act.mean()]*len(act)); r_ub=rmse(ub_p); r_ib=rmse(ib_p); r_cb=rmse(cb_p)
p_bl=prec([act.mean()]*len(act)); p_ub=prec(ub_p); p_ib=prec(ib_p); p_cb=prec(cb_p)

print(f'\n{"Methode":<20} {"RMSE":>8} {"Precision":>12}')
print('-'*42)
for name,r,p in [("Baseline",r_bl,p_bl),("User-Based",r_ub,p_ub),
                  ("Item-Based",r_ib,p_ib),("Content-Based",r_cb,p_cb)]:
    print(f'{name:<20} {r:>8.4f} {p:>11.1%}')

best = min([('User-Based',r_ub),('Item-Based',r_ib),('Content-Based',r_cb)], key=lambda x:x[1])
print(f'\nMeilleure approche : {best[0]} (RMSE={best[1]:.4f})')

In [ ]:
labels = ['Baseline','User-Based','Item-Based','Content-Based']
rmses  = [r_bl, r_ub, r_ib, r_cb]
precs  = [p_bl, p_ub, p_ib, p_cb]
colors = ['#444466', ACCENT, BLUE, GREEN]

fig, axes = plt.subplots(1,2,figsize=(14,5))
fig.suptitle('Comparaison des approches', color='#e0e0f0', fontsize=13)

for ax,(vals,title,pct) in zip(axes,[
    (rmses,'RMSE (bas = meilleur)',False),
    (precs,'Precision +/-0.5 (haut = meilleur)',True)]):
    bars = ax.bar(labels, vals, color=colors, alpha=0.85)
    ax.set_title(title, color='#e0e0f0')
    if pct: ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_ylim(0, max(vals)*1.25)
    for b,v in zip(bars,vals):
        label = f'{v:.1%}' if pct else f'{v:.3f}'
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+max(vals)*0.02,
                label, ha='center', va='bottom', fontsize=9, color='#e0e0f0')

plt.tight_layout(); plt.show()

## Etape 7 — Contraintes techniques

In [ ]:
# --- Contrainte 1 : Similarite cosinus obligatoire ---
# Demonstration sur deux utilisateurs
u1 = matrix_filled.iloc[0].values
u2 = matrix_filled.iloc[1].values

cosine_manuel = np.dot(u1, u2) / (np.linalg.norm(u1) * np.linalg.norm(u2))
cosine_sklearn = cosine_similarity([u1], [u2])[0][0]

print('Verification similarite cosinus (manuel vs sklearn) :')
print(f'  Manuel  : {cosine_manuel:.6f}')
print(f'  Sklearn : {cosine_sklearn:.6f}')
print(f'  Identiques : {abs(cosine_manuel - cosine_sklearn) < 1e-9}')

In [ ]:
# --- Contrainte 2 : Exemple concret avec un utilisateur reel ---
DEMO_USER = 42  # utilisateur reel du dataset

films_vus = ratings[ratings.user_id == DEMO_USER].merge(
    movies[['item_id', 'title', 'genres_str']], on='item_id'
).sort_values('rating', ascending=False)

print(f'Utilisateur {DEMO_USER} — {len(films_vus)} films notes | Moyenne : {films_vus.rating.mean():.2f}/5')
print('\nTop 5 films les mieux notes par cet utilisateur :')
display(films_vus[['title', 'rating', 'genres_str']].head(5))

reco_demo_ub = user_based_recommend(DEMO_USER)
reco_demo_ib = item_based_recommend(DEMO_USER)

print('\nRecommandations User-Based :')
display(reco_demo_ub[['title', 'genres_str', 'note_predite']])

print('\nRecommandations Item-Based :')
display(reco_demo_ib[['title', 'genres_str', 'note_predite']])

In [ ]:
# --- Contrainte 3 : Comparaison User-Based vs Item-Based ---
print(f'{"Critere":<30} {"User-Based":>15} {"Item-Based":>15}')
print('-' * 62)
print(f'{"RMSE":<30} {r_ub:>15.4f} {r_ib:>15.4f}')
print(f'{"Precision (+/-0.5)":<30} {p_ub:>14.1%} {p_ib:>14.1%}')
print(f'{"Similarite utilisee":<30} {"cosinus":>15} {"cosinus":>15}')
print(f'{"Dimension matrice":<30} {"users x users":>15} {"items x items":>15}')
print(f'{"Voisins consideres":<30} {"20":>15} {"10":>15}')

meilleur = 'User-Based' if r_ub < r_ib else 'Item-Based'
print(f'\nMeilleure approche (RMSE) : {meilleur}')

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('User-Based vs Item-Based', color='#e0e0f0', fontsize=13)

axes[0].bar(['User-Based', 'Item-Based'], [r_ub, r_ib], color=[ACCENT, BLUE], alpha=0.85)
axes[0].set_title('RMSE (bas = meilleur)', color='#e0e0f0')
for i, v in enumerate([r_ub, r_ib]):
    axes[0].text(i, v + 0.01, f'{v:.4f}', ha='center', va='bottom', color='#e0e0f0', fontsize=10)

axes[1].bar(['User-Based', 'Item-Based'], [p_ub, p_ib], color=[ACCENT, BLUE], alpha=0.85)
axes[1].set_title('Precision +/-0.5 (haut = meilleur)', color='#e0e0f0')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
for i, v in enumerate([p_ub, p_ib]):
    axes[1].text(i, v + 0.005, f'{v:.1%}', ha='center', va='bottom', color='#e0e0f0', fontsize=10)

plt.tight_layout(); plt.show()

## Conclusion

| Point | Realise |
|---|---|
| 1. Import MovieLens local | data/u.data + data/u.item |
| 2. Matrice User-Movie | pivot_table + heatmap |
| 3a. User-Based (cosine) | OK |
| 3b. Item-Based (cosine) | OK |
| 4. Content-Based (genres) | OK |
| 5. Top-5 utilisateur reel | USER_ID configurable |
| 6. RMSE + Precision + Comparaison | Train/Test 80/20 |
| 7. Contraintes techniques | cosinus + exemple reel + comparaison |